# 03 MLM Baseline Pre-training
This notebook trains a standard RoBERTa-base model using Masked Language Modeling (MLM) on the ECtHR dataset using the HuggingFace `Trainer`.

In [1]:
import sys
import os
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast, 
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments,
)

# --- CONFIGURATION ---
CHECKPOINT_DIR = "../checkpoints/mlm"
EPOCHS         = 5
BATCH_SIZE     = 8  # 24GB-32GB VRAM. Scale up if on Spark.
WANDB_PROJECT  = "glocal-nlp"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [2]:
print("Loading and flattening dataset...")
raw_dataset = load_dataset("coastalcph/lex_glue", "ecthr_a")
raw_dataset = raw_dataset.filter(lambda x: len(x["text"]) >= 5)

def flatten_paragraphs(example):
    return {"text": " ".join(example["text"])}

dataset = raw_dataset.map(flatten_paragraphs, remove_columns=["labels"])
print(f"Train size: {len(dataset['train'])}")

Loading and flattening dataset...


Filter:   0%|          | 0/9000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/992 [00:00<?, ? examples/s]

Train size: 8988


In [ ]:
print("Tokenizing...")
tokenizer = RobertaTokenizerFast.from_pretrained("distilroberta-base")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
tokenized_dataset.set_format("torch")

In [ ]:
print("Initializing Trainer...")
model = RobertaForMaskedLM.from_pretrained("distilroberta-base")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    save_steps                  = 1000,
    logging_steps               = 100,
    report_to                   = "wandb",
    run_name                    = "mlm_baseline",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset["train"],
    data_collator = data_collator,
)

print("Starting MLM Training...")
trainer.train()
trainer.save_model(f"{CHECKPOINT_DIR}/mlm_final")
print("MLM Training Complete.")